# Methodology Walkthrough

A narrative tour of how the Restaurant Location Intelligence Tool turns a city, a cuisine and a budget into a ranked list of zones. Readable without a technical background.


## 1. The idea

Every restaurant operator eventually asks: *where should we open the next one?* The usual answer is a mix of broker intros, walking the neighbourhood, and gut. This notebook walks through a reproducible alternative: slice the city into equal-area hexagons, score each one, and rank the top candidates.


## 2. Why hexagons

Administrative neighbourhoods have ragged, inconsistent sizes - comparing them is apples to oranges. [Uber's H3 library](https://h3geo.org) covers the globe in equal-area hexagons at nested resolutions. At resolution 8 each hex is about 0.74 km2 - small enough to see the difference between Gracia and Eixample, large enough to carry meaningful POI counts.


## 3. The six scores

For each hex we compute six numbers in [0, 1]:

- **Competition gap** - how many same-cuisine restaurants are nearby. Zero is great, 5+ is a problem.
- **Foot traffic** - a mix of shops, transit stops, offices and nightlife.
- **Income match** - does the area's household income fit the cuisine's price point? A fine-dining omakase and a budget taqueria do not want the same block.
- **Tourism intensity** - hotel beds and tourist attractions. Matters a lot for ethnic cuisines like Indian, Japanese, Thai.
- **Restaurant ecosystem** - a bell curve on total restaurants. Zero restaurants is a dead zone, 30+ is saturated, around 8-12 is a healthy dining strip.
- **Rent affordability** - estimated commercial rent, scored against the operator's budget tier.


## 4. Combining them

Each component is weighted by cuisine. A burger joint lives on foot traffic; an omakase lives on income match and tourism. Weights sum to 1.0 and the final composite is scaled to 0-100.


In [ ]:
import sys
sys.path.insert(0, '..')
from src.pipeline import run_pipeline

df = run_pipeline('barcelona', 'indian', 'medium')
df.sort_values('composite', ascending=False).head(10)[
    ['h3_id', 'composite', 'competition_gap', 'foot_traffic',
     'income_match', 'tourism_intensity', 'restaurant_ecosystem',
     'rent_affordability']
]

## 5. What the model cannot see

- Real pedestrian counts. We proxy with static POIs.
- Live rent listings. We estimate from income and traffic.
- Street-level nuance: a corner premium, a metro entrance 40m away, a competitor's lease expiring next quarter.

The model is a filter, not a decision. Use it to shortlist 10 candidates from 1,000, then walk the ones that score well.
